# Wrap a live game as an MCP server

By the end of this notebook Claude will be building bridges in the **pygame window
beside you**, in real time, and choosing them the way a client asks.

**Layout (keep all three visible):**

1. This notebook
2. The Bridge Builder window
3. Claude Code

The server grows in three stages. Each stage adds one kind of MCP building block,
and each time we ask Claude **the same thing**: *"Build me the cheapest bridge."*
Watch how the answer changes.

| Stage | Adds | File | What the model can do |
|---|---|---|---|
| 1 | **Tools**, written by you | `my_tools.py` | act: look, build, test |
| 2 | **Resources**, written by you | `my_resources.py` | know: level facts, tested designs |
| 3 | **Prompts**: a design, a client brief | `mcp_clients.py` | follow a workflow and a client's requirements |

## 0. Start the game at stage 1

In a terminal, from the repo root (already `uv sync`'d):

```bash
uv run python -m bridge_builder --stage 1
```

Red **anchor** circles on both banks. Click an anchor, drag to a neighbour.
**Space** sends the train. **R** resets. Leave the window running.

The terminal says `tools from my_tools.py: none yet`. The MCP server is running, but
it is bare: a name, instructions and a line to the game. Run `/mcp` in Claude Code:
`bridge-engineer` is connected and offers **no tools**. Claude can read about the
level and do nothing in it. The tools are yours to write.

To move to the next stage later: close the window, start it again with the next
`--stage`, then run `/mcp` in Claude Code to reconnect. Stages 2 and 3 serve the
reference tools, so everyone starts them from the same working server (`--mine`
keeps yours). Stuck in stage 1? `--solution` serves the reference tools.

## Stage 1: tools, so the model can act

A tool is an ordinary Python function plus `@mcp.tool()`. The SDK publishes the
function name, its docstring and its type hints as the tool the model sees.

This is your file for today, `src/bridge_builder/my_tools.py`. The functions are
written; you add the decorators. Open it in the editor; the cell below only prints it.

In [ ]:
from pathlib import Path
import bridge_builder

# Read, not imported: the file may be half-written while you work on it.
MY_TOOLS = Path(bridge_builder.__file__).parent / "my_tools.py"
print(MY_TOOLS.read_text(encoding="utf-8"))

### Together: `place_girder`

The functions in `my_tools.py` are written already: each sends a command to the
game and returns its answer. But a plain function is invisible to the model. One
line above `def place_girder` makes it a tool:

```python
@mcp.tool(annotations=ToolAnnotations(destructive_hint=False, open_world_hint=False))
```

The SDK now publishes its name, its docstring and its type hints as the tool the
model sees; the annotations say it adds and never deletes, and touches only the game.

Then check it the way the model will call it, with no LLM and no window:

In [ ]:
!uv run workshop check

When `place_girder` shows **ok**, close the game window and start it again with
`--stage 1`. The terminal now lists `place_girder`.

### Ping the game and register it with Claude

`register_claude()` writes `.mcp.json` in this project (and the Claude Desktop
config if that app is installed). Claude Code picks up project MCP servers from
that file.

In [ ]:
from bridge_builder.client import (
    call_tool,
    get_prompt,
    list_prompts,
    list_resources,
    list_tools,
    read_resource,
    register_claude,
)

print("tools:", list_tools())
register_claude()

If Claude Code is already open on this folder, run `/mcp` (or reconnect) **once**.
`bridge-engineer` should then be listed as an HTTP server at
`http://127.0.0.1:8765/mcp`, now with one tool.

### Drive the game with the same tool the model will use

Build a **naive straight deck** at `y=3`. Watch the beams appear in the window.

In [ ]:
for x in range(-6, 6, 2):
    print(x, "->", x + 2, call_tool("place_girder", x1=x, y1=3, x2=x + 2, y2=3))

Claude can build now, but it cannot test what it built, clear the level or see
the result.

## Your turn: `start_train`, `reset`, `status`

Add the decorator above each of them in `my_tools.py`, under *Your turn*. The
comments hint at the right `ToolAnnotations`. Run the check after each one. Fast?
Carry on with the stretch tools: `place_girders`, `destroy_at`, `pause`.

In [ ]:
!uv run workshop check

When all four are **ok**: restart the game with `--stage 1` and run the next cell.
It builds the straight deck again (a restart clears the level) and sends the train.

In [ ]:
for x in range(-6, 6, 2):
    call_tool("place_girder", x1=x, y1=3, x2=x + 2, y2=3)
call_tool("start_train")

Watch the window. A straight line of pinned beams is a chain: it sags, the beams
glow red, and one snaps. A line is not a bridge.

In [ ]:
print(call_tool("status"))
call_tool("reset")

### Ask Claude

In Claude Code, run `/mcp` to pick up your new tools, then:

> Build me the cheapest bridge.

It has your tools but no knowledge of this level: it improvises. It may build a
truss, it may forget a diagonal and watch the bridge fold. Note what it built and
what it cost. Then look at how it used your tools: did it call `status` after the
train? A better docstring changes what the model does.

## Stage 2: resources, so the model can know

A resource is read-only content the model can pull into context, published with
`@mcp.resource(uri)`. A URI with `{placeholders}` is a template: one function serves
a whole family of URIs.

The knowledge is ready: four tested reference designs in `knowledge/designs/` (a card
and the exact beams for each). Your file for this stage is
`src/bridge_builder/my_resources.py`: the functions that serve them are written, you add
`@mcp.resource(uri, mime_type=...)` above each. The catalog together, then the design
card, its beams and the level facts. Check with `uv run workshop check --stage 2`.

Then restart the game with `--stage 2` and `/mcp` in Claude Code. Stage 2 also adds
server instructions that point the model at the designs, and a `build_design` tool,
and serves the reference tools, so everyone starts it from the same working stage 1.

In [ ]:
from pathlib import Path
import bridge_builder

# Read, not imported: the file may be half-written while you work on it.
MY_RESOURCES = Path(bridge_builder.__file__).parent / "my_resources.py"
print(MY_RESOURCES.read_text(encoding="utf-8"))

In [ ]:
print("resources:", list_resources())
print(read_resource("bridge://designs"))

### Ask Claude the same thing

> Build me the cheapest bridge.

The server instructions now tell the model to offer a choice first: a reference
design or a custom one. Ask for the cheapest and it builds the **inverted truss**
(2100). It ties with the overhead truss on cost but has far more margin, and the
catalog lists it first.

## Stage 3: prompts, workflows the user starts

Restart the game with `--stage 3`, then `/mcp` in Claude Code.

A prompt is a reusable workflow with arguments, published with `@mcp.prompt()`; in
Claude Code it becomes a slash command. The first one is simple:
`/bridge-engineer:design inverted_truss` hands the model that design's card and exact
beams, and the steps to build and test it.

Your file for this stage is `src/bridge_builder/my_prompts.py`: both prompts are
written, you add `@mcp.prompt(name=..., title=...)` above each, `design` together and
`brief` yourself. Check with `uv run workshop check --stage 3`.

The tools and the designs know nothing about **who** the bridge is for. FETNIS has
requirements: ships must pass under the bridge, and it should be as sustainable as
possible. That knowledge lives in a brief, a Markdown file in `knowledge/clients/`,
published as a resource. The `brief` prompt puts the brief and the design catalog
in front of the model with one job: build what the client's requirements rank highest.

In [ ]:
from pathlib import Path
import bridge_builder

# Read, not imported: the file may be half-written while you work on it.
MY_PROMPTS = Path(bridge_builder.__file__).parent / "my_prompts.py"
print(MY_PROMPTS.read_text(encoding="utf-8"))

In [ ]:
print(read_resource("bridge://clients/fetnis"))

This is exactly what the model receives when you run the prompt:

In [ ]:
print(get_prompt("brief", client="fetnis"))

### Ask Claude, as FETNIS

> /bridge-engineer:brief fetnis

The cheapest design, the inverted truss, hangs below the road: ships cannot pass,
so the brief rules it out. Of the rest, the **timber arch** has by far the highest
wood share (18 of 28 beams). It costs 3160, over 1000 more than the cheapest, and
the model should tell you so.

**Where does knowledge belong?**

| Put it in | Reaches the model | Good for |
|---|---|---|
| Server instructions | every conversation | rules of the game |
| Resources | when the model (or you) reads them | facts, designs, briefs |
| Prompts | when **you** invoke them | a workflow, one client's requirements |

A prompt is a policy the model follows, not a rule the server enforces. If
clearance were a legal requirement, where would you put it so it *cannot* be broken?

## Exercise: write your own brief

Briefs are re-read on every request, so a new file works **without a restart**.
Run the cell, edit the file it prints, then in Claude Code:

> /bridge-engineer:brief harbour

Ideas: a harbour authority that only needs clearance and the lowest price; a city
that bans titanium; a low-profile bridge with nothing above `y=5`.

In [ ]:
from bridge_builder.clients import CLIENTS_DIR

brief = CLIENTS_DIR / "harbour.md"
brief.write_text("""# Harbour Authority

The cheapest bridge that ships can sail under.

## Requirements, in priority order

1. **Clearance (must).** No beam may hang below the deck: every beam inside the
   gap (x between -6 and 6) must have both ends at y >= 3.
2. **Cost.** The cheapest design that meets 1.
3. **Safety.** Peak beam load below 0.9.

## Report

The chosen design, its cost and peak load, and why the cheapest design overall
was or was not chosen.
""", encoding="utf-8")
print(brief)
print(read_resource("bridge://clients"))

## Reference: materials

`place_girder` takes an optional `material`. The level resource lists the table:

| material | cost | break force (pull) | stiffness | density |
|---|---|---|---|---|
| steel | 100 | 1.0x (950 N) | 1.0 | 1.0 |
| wood | 120 | 0.6x (570 N) | 0.5 | 0.5 |
| titanium | 400 | 2.0x (1900 N) | 0.8 | 0.8 |

Steel is the cheap default. Wood costs more and is weaker: you choose it for
sustainability. A beam breaks when its pull or push is too much; long diagonals
buckle under push at half the load of a 2 m beam. Joints turn freely, so only
triangles are rigid.

The grid also continues **below** the deck inside the gap (`x=-4..4`, `y=1` and
`y=-1`), so an inverted truss hanging under the road is a valid design. The bank
walls at `x=±6` are solid below the anchors.

## Spoiler

Working trusses (hide until someone is stuck). Importing them does **not** build
them: Claude still has to place each beam in order from an existing node.

In [ ]:
from bridge_builder.truss import INVERTED_TRUSS, TRUSS_GIRDERS

print("overhead truss (steel, 2100):")
for row in TRUSS_GIRDERS:
    print(row)

print("\ninverted, under the deck (steel, 2100):")
for row in INVERTED_TRUSS:
    print(row)